🌐 Django Tutorial › More Django &nbsp;·&nbsp; *Bài 46/47*

[« Django - Deploy](045_django_deploy.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../../README.md) &nbsp;•&nbsp; [Django - Add Bootstrap 5 »](047_django_add_bootstrap5.ipynb)

# Django - Add Slug Field

> 📖 **Học song song trên W3Schools:** [django_slug_field.php](https://www.w3schools.com/django/django_slug_field.php)
>
> 🎯 **Trong bài này:** What is Slug? · Add a Slug Field · Change Admin · Enter Slug Values · Use Slug in URLs · 👀 Xem kết quả

## What is Slug? — Slug là gì?

URL trang chi tiết hiện có dạng `/members/details/1`. Số `1` là `id` trong cơ sở dữ liệu — **không** có ý nghĩa với người dùng hay công cụ tìm kiếm.

**Slug** là một chuỗi chỉ gồm chữ thường, số, gạch ngang — mô tả nội dung, ví dụ `/members/details/an-nguyen`. URL dạng này **dễ đọc** và **tốt cho SEO**.

## Add a Slug Field — Thêm trường slug

In [1]:
%%writefile my_tennis_club/members/models.py
from django.db import models

class Member(models.Model):
    firstname = models.CharField(max_length=255)
    lastname = models.CharField(max_length=255)
    phone = models.IntegerField(null=True)
    joined_date = models.DateField(null=True)
    slug = models.SlugField(default="", null=False)

    def __str__(self):
        return f"{self.firstname} {self.lastname}"

Overwriting my_tennis_club/members/models.py


In [2]:
!cd my_tennis_club && python manage.py makemigrations members

Migrations for 'members':
  members\migrations\0003_member_slug.py
    + Add field slug to member


In [3]:
!cd my_tennis_club && python manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, contenttypes, members, sessions
Running migrations:
  Applying members.0003_member_slug... OK


## Change Admin — Tự điền slug trong admin

Trong admin, dùng `prepopulated_fields` để slug **tự sinh** từ tên khi nhập:

In [4]:
%%writefile my_tennis_club/members/admin.py
from django.contrib import admin
from .models import Member

# Register your models here.

class MemberAdmin(admin.ModelAdmin):
    list_display = ("firstname", "lastname", "joined_date",)
    prepopulated_fields = {"slug": ("firstname", "lastname")}

admin.site.register(Member, MemberAdmin)

Overwriting my_tennis_club/members/admin.py


## Enter Slug Values — Điền slug cho dữ liệu cũ

W3Schools mở từng thành viên trong admin và bấm SAVE để slug được điền. Ta làm nhanh bằng code với hàm `slugify()` (tự bỏ dấu tiếng Việt):

In [5]:
from django_helper import setup

setup()
from django.utils.text import slugify
from members.models import Member

print(slugify("An Nguyễn"))
print(slugify("Hà Đặng"))   # ⚠️ mất chữ "Đ"!

an-nguyen
ha-ang


> ⚠️ **Bẫy với tiếng Việt:** `slugify()` bỏ dấu bằng cách tách ký tự gốc và dấu (Unicode NFKD). Nhưng **"đ/Đ"** là một chữ cái riêng, không phải "d" + dấu — nên bị **xoá mất**. Cách xử lý: đổi "đ" thành "d" trước khi slugify.

In [6]:
def vn_slugify(text):
    return slugify(text.replace("đ", "d").replace("Đ", "D"))

for m in Member.objects.all():
    m.slug = vn_slugify(f"{m.firstname} {m.lastname}")
    m.save()

print(Member.objects.values_list("firstname", "lastname", "slug"))

<QuerySet [('An', 'Nguyễn', 'an-nguyen'), ('Bình', 'Trần', 'binh-tran'), ('Chi', 'Lê', 'chi-le'), ('Dũng', 'Phạm', 'dung-pham'), ('Em', 'Hoàng', 'em-hoang'), ('Giang', 'Vũ', 'giang-vu'), ('Hà', 'Đặng', 'ha-dang')]>


## Use Slug in URLs — Dùng slug trong URL

Đổi URL, view và template để dùng slug thay cho id:

In [7]:
%%writefile my_tennis_club/members/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path('', views.main, name='main'),
    path('members/', views.members, name='members'),
    path('members/details/<slug:slug>', views.details, name='details'),
    path('testing/', views.testing, name='testing'),
]

Overwriting my_tennis_club/members/urls.py


In [8]:
from django_helper import replace_view

replace_view("details", '''
def details(request, slug):
    mymember = Member.objects.get(slug=slug)
    template = loader.get_template('details.html')
    context = {
        'mymember': mymember,
    }
    return HttpResponse(template.render(context, request))
''')

✔ Đã cập nhật view details() trong my_tennis_club/members/views.py


In [9]:
%%writefile my_tennis_club/members/templates/all_members.html
{% extends "master.html" %}

{% block title %}
  My Tennis Club - List of all members
{% endblock %}


{% block content %}
  <div class="mycard">
    <h1>Members</h1>
    <ul>
      {% for x in mymembers %}
        <li><a href="details/{{ x.slug }}">{{ x.firstname }} {{ x.lastname }}</a></li>
      {% endfor %}
    </ul>
  </div>
{% endblock %}

Overwriting my_tennis_club/members/templates/all_members.html


## 👀 Xem kết quả

In [10]:
from django_helper import preview

preview("/members/", max_chars=1200)
preview("/members/details/an-nguyen", max_chars=1200)

GET /members/  ->  200

<!DOCTYPE html>
<html>
<head>
  <title>
  My Tennis Club - List of all members
</title>
  <link rel="stylesheet" href="/static/mystyles.css">
</head>
<body>

<div class="topnav">
  <a href="/">HOME</a>
  <a href="/members">Members</a>
</div>

<div class="main">
  
  <div class="mycard">
    <h1>Members</h1>
    <ul>
      
        <li><a href="details/an-nguyen">An Nguyễn</a></li>
      
        <li><a href="details/binh-tran">Bình Trần</a></li>
      
        <li><a href="details/chi-le">Chi Lê</a></li>
      
        <li><a href="details/dung-pham">Dũng Phạm</a></li>
      
        <li><a href="details/em-hoang">Em Hoàng</a></li>
      
        <li><a href="details/giang-vu">Giang Vũ</a></li>
      
        <li><a href="details/ha-dang">Hà Đặng</a></li>
      
    </ul>
  </div>

</div>

</body>
</html>

GET /members/details/an-nguyen  ->  200

<!DOCTYPE html>
<html>
<head>
  <title>
  Details about An Nguyễn
</title>
  <link rel="stylesheet" href="/static/mys

---

## 🧠 Ghi nhớ nhanh

> - Slug = chuỗi thân thiện URL (`an-nguyen`) — dễ đọc, tốt cho SEO.
> - `models.SlugField()` + `prepopulated_fields` trong admin + `slugify()` để tạo slug.
> - ⚠️ `slugify()` xoá mất chữ **đ/Đ** → thay bằng d/D trước.
> - URL: `path('members/details/<slug:slug>', ...)` → view `details(request, slug)`.

---

## 📝 Exercise — Bài tập nhanh

**❓ Câu hỏi:** `slugify("An Nguyễn")` cho kết quả là gì?

- **A.** `"An Nguyễn"`
- **B.** `"AN-NGUYEN"`
- **C.** `"an_nguyễn"`
- **D.** `"an-nguyen"`

<details>
<summary>👉 Xem đáp án</summary>

✅ **Đáp án: D** — `"an-nguyen"`

</details>

---

[« Django - Deploy](045_django_deploy.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../../README.md) &nbsp;•&nbsp; [Django - Add Bootstrap 5 »](047_django_add_bootstrap5.ipynb)